# bge-reranker-v2-m3 LoRA 微调 (Google Colab T4 GPU)

【不易】不修改原训练脚本,仅通过 `!python` 调用 `finetune_reranker.py`
【变易】支持从本地上传或从 GitHub 拉取训练数据
【简易】单 notebook 完成:环境检查 → 依赖安装 → 数据上传 → 训练 → 下载模型

**目标**: 在 Colab 免费 T4 GPU (15GB) 上完成 567M 参数 reranker 的 LoRA 微调

**预计耗时**: 5-10 分钟(含模型下载 + 5 epoch 训练)

---

## 步骤 1: 检查 GPU

确认 Colab 已分配 T4 GPU(菜单栏 → 代码执行程序 → 更改运行时类型 → T4 GPU)

In [ ]:
# 检查 NVIDIA 驱动 + GPU 型号
!nvidia-smi

print("\n=" * 40)

# 检查 PyTorch CUDA 可用性
import torch
print(f"\nPyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"显存: {mem_gb:.1f} GB")
    assert mem_gb >= 14, f"显存不足({mem_gb:.1f}GB < 14GB),请改用 T4 或更高 GPU"
    print("\n✅ GPU 检查通过")
else:
    raise RuntimeError("\n❌ CUDA 不可用,请在菜单栏选择 GPU 运行时")

## 步骤 2: 安装依赖

Colab 预装了 PyTorch,但需要补充 PEFT / accelerate / sentence-transformers

In [ ]:
# 升级 pip 并安装训练依赖
!pip install -q --upgrade pip wheel
!pip install -q "peft>=0.7.0" "accelerate>=0.27.0" \
    "sentence-transformers>=2.3.0" "transformers>=4.38.0" \
    "scikit-learn>=1.3.0" "numpy<2.0" "tqdm>=4.66.0" \
    "huggingface_hub>=0.20.0"

print("\n=" * 40)
print("依赖安装完成,版本检查:")
import peft, accelerate, sentence_transformers, transformers, numpy
print(f"  peft: {peft.__version__}")
print(f"  accelerate: {accelerate.__version__}")
print(f"  sentence-transformers: {sentence_transformers.__version__}")
print(f"  transformers: {transformers.__version__}")
print(f"  numpy: {numpy.__version__} (必须 < 2.0)")
assert numpy.__version__.split('.')[0] < '2', "numpy 必须 < 2.0"
print("\n✅ 依赖检查通过")

## 步骤 3: 设置工作目录 + HF 镜像

- 使用 `/content/` 作为工作根目录
- 配置 HF 镜像(`hf-mirror.com`)加速模型下载

In [ ]:
import os

# 切换到 Colab 标准工作目录
%cd /content/

# 创建项目目录结构(模拟本地 c:\Users\Administrator\agent 布局)
!mkdir -p agent/scripts/deploy_gpu_training
!mkdir -p agent/data

# 配置 HF 镜像(国内镜像,海外 Colab 也可访问)
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '0'
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
# 减少显存碎片
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

print("✅ 工作目录: /content/agent")
print("✅ HF 镜像: https://hf-mirror.com")
print("✅ 环境变量已设置")

## 步骤 4: 上传训练脚本与数据

**方式 A**(推荐): 从本地上传 `finetune_reranker.py` 和训练数据

**方式 B**: 从 GitHub 克隆(若项目已推送)

In [ ]:
# === 方式 A: 从本地上传文件 ===
# 点击运行后会弹出文件选择框,依次选择:
#   1. c:\Users\Administrator\agent\scripts\finetune_reranker.py
#   2. c:\Users\Administrator\agent\data\reranker_trainset.jsonl
#   3. c:\Users\Administrator\agent\data\reranker_valset.jsonl

from google.colab import files
import shutil
from pathlib import Path

print("📤 请选择 finetune_reranker.py ...")
uploaded = files.upload()
for fname in uploaded.keys():
    if fname == 'finetune_reranker.py' or 'finetune_reranker' in fname:
        shutil.copy(fname, '/content/agent/scripts/finetune_reranker.py')
        print(f"  ✅ 已上传: scripts/finetune_reranker.py ({len(uploaded[fname])} bytes)")

print("\n📤 请选择 reranker_trainset.jsonl ...")
uploaded = files.upload()
for fname in uploaded.keys():
    if 'trainset' in fname:
        shutil.copy(fname, '/content/agent/data/reranker_trainset.jsonl')
        print(f"  ✅ 已上传: data/reranker_trainset.jsonl ({len(uploaded[fname])} bytes)")

print("\n📤 请选择 reranker_valset.jsonl ...")
uploaded = files.upload()
for fname in uploaded.keys():
    if 'valset' in fname:
        shutil.copy(fname, '/content/agent/data/reranker_valset.jsonl')
        print(f"  ✅ 已上传: data/reranker_valset.jsonl ({len(uploaded[fname])} bytes)")

In [ ]:
# === 方式 B: 从 GitHub 克隆(若项目已推送,取消注释并替换 URL) ===
# !cd /content && git clone https://github.com/<USER>/<REPO>.git agent
# !cp -r /content/agent/scripts/finetune_reranker.py /content/agent/scripts/ 2>/dev/null || true

In [ ]:
# 校验文件就位
%cd /content/agent

import os
required = [
    'scripts/finetune_reranker.py',
    'data/reranker_trainset.jsonl',
    'data/reranker_valset.jsonl',
]
for f in required:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  ✅ {f} ({size} bytes)")
    else:
        print(f"  ❌ {f} 缺失!")
        raise FileNotFoundError(f)

# 统计样本数
with open('data/reranker_trainset.jsonl', 'r', encoding='utf-8') as f:
    train_n = sum(1 for _ in f)
with open('data/reranker_valset.jsonl', 'r', encoding='utf-8') as f:
    val_n = sum(1 for _ in f)
print(f"\n训练集: {train_n} 样本")
print(f"验证集: {val_n} 样本")

## 步骤 5: 预下载基础模型

提前下载 `BAAI/bge-reranker-v2-m3`(约 2.2GB),避免训练时网络中断

In [ ]:
from huggingface_hub import snapshot_download

print("⏬ 预下载模型: BAAI/bge-reranker-v2-m3 (约 2.2GB)")
model_path = snapshot_download('BAAI/bge-reranker-v2-m3')
print(f"\n✅ 模型已缓存到: {model_path}")

## 步骤 6: 启动训练

**关键**: 使用 `!python` 调用,输出实时显示在 notebook 中

训练参数:
- `--epochs 5`: 最多 5 个 epoch
- `--batch-size 16`: T4 16GB 显存可容纳
- `--lora-rank 8 --lora-alpha 2`: LoRA 配置(仅训练 0.32% 参数)
- `--optimizer adamw`: GPU 模式(9GB state)
- `--early-stopping-patience 2`: 连续 2 个 epoch 无改善则早停

预计耗时: 约 2-3 分钟/epoch,总计 10-15 分钟

In [ ]:
# 切换到项目根目录
%cd /content/agent

# 启动训练(!python 调用,输出实时显示)
!python scripts/finetune_reranker.py \
    --train data/reranker_trainset.jsonl \
    --val data/reranker_valset.jsonl \
    --output data/reranker_finetuned/ \
    --base-model BAAI/bge-reranker-v2-m3 \
    --max-length 512 \
    --epochs 5 \
    --batch-size 16 \
    --lr 2e-5 \
    --lora-rank 8 \
    --lora-alpha 2 \
    --early-stopping-patience 2 \
    --optimizer adamw

## 步骤 7: 检查训练结果

查看输出目录 + 训练元信息

In [ ]:
import os, json
from pathlib import Path

output_dir = Path('/content/agent/data/reranker_finetuned')
print(f"输出目录: {output_dir}\n")

if output_dir.exists():
    print("文件清单:")
    for f in sorted(output_dir.iterdir()):
        size = f.stat().st_size
        size_str = f"{size/1024/1024:.2f} MB" if size > 1024*1024 else f"{size/1024:.1f} KB"
        print(f"  {f.name:40s} {size_str}")

    meta_file = output_dir / 'training_meta.json'
    if meta_file.exists():
        print("\n训练元信息:")
        with open(meta_file, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        for k, v in meta.items():
            if k == 'val_accuracy':
                print(f"  {k}: {v:.2%}")
            elif k == 'train_time_sec':
                print(f"  {k}: {v:.1f}s ({v/60:.1f}min)")
            else:
                print(f"  {k}: {v}")
else:
    print("❌ 输出目录不存在,训练可能失败")

## 步骤 8: 打包并下载模型到本地

将微调后的模型打包为 `.tar.gz`,通过浏览器下载到本地 Windows 机器

In [ ]:
# 打包模型(tar.gz)
!cd /content/agent/data && tar -czf /content/reranker_finetuned.tar.gz reranker_finetuned

import os
size_mb = os.path.getsize('/content/reranker_finetuned.tar.gz') / (1024*1024)
print(f"✅ 打包完成: /content/reranker_finetuned.tar.gz ({size_mb:.2f} MB)")

In [ ]:
# 下载到本地(浏览器自动触发下载)
from google.colab import files
files.download('/content/reranker_finetuned.tar.gz')

## 步骤 9: 本地解压并使用

下载到本地后,在 Windows PowerShell 中执行:

```powershell
# 假设下载到 C:\Users\Administrator\Downloads\
cd C:\Users\Administrator\agent\data
tar -xzf C:\Users\Administrator\Downloads\reranker_finetuned.tar.gz

# 验证模型
python scripts\eval_reranker_zero_shot.py --model data\reranker_finetuned
```

模型即可用于本地推理,无需 GPU。

## 故障排查

### 问题 1: CUDA out of memory
```python
# 降低 batch_size 或 max_length
!python scripts/finetune_reranker.py \
    --train data/reranker_trainset.jsonl \
    --val data/reranker_valset.jsonl \
    --output data/reranker_finetuned/ \
    --batch-size 8 \           # 从 16 降到 8
    --max-length 384 \          # 从 512 降到 384
    --optimizer adamw
```

### 问题 2: 模型下载失败
```python
# 使用 HF 镜像重新下载
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
from huggingface_hub import snapshot_download
snapshot_download('BAAI/bge-reranker-v2-m3', max_workers=4)
```

### 问题 3: 训练被中断(Colab 12 小时限制)
- 使用较小 epoch 数(如 3)
- 或分多次运行,每次从 `data/reranker_finetuned/` 继续训练

### 问题 4: 验证集准确率未提升
- 检查训练数据正负样本比例
- 调整学习率(尝试 `--lr 1e-5` 或 `--lr 5e-5`)
- 增大 LoRA rank(`--lora-rank 16 --lora-alpha 32`)